# Composed strategy evaluation — high-frequency bucket

Runs the top-5 high-frequency combos (selected by ML#1 composite-score)
on the 20% chronological test partition. Reports per-combo performance
and overlays equity curves + drawdown.

**Prerequisite:** generate `evaluation/top_strategies.json` first via
`python scripts/runners/run_extract_top_combos.py`.

In [ ]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

REPO = Path.cwd().resolve()
while not (REPO / 'src').exists() and REPO.parent != REPO:
    REPO = REPO.parent
sys.path.insert(0, str(REPO))

from scripts.evaluation.composed_strategy_runner import run_strategy, load_test_bars

BUCKET = 'high_freq'
TOP_STRATEGIES_PATH = REPO / 'evaluation' / 'top_strategies.json'
STARTING_EQUITY = 50_000.0

In [ ]:
strategies = json.loads(TOP_STRATEGIES_PATH.read_text())[BUCKET]
print(f'Loaded {len(strategies)} {BUCKET} strategies')
bars = load_test_bars()
print(f'Test partition: {len(bars):,} bars  {bars["time"].iloc[0]} -> {bars["time"].iloc[-1]}')

In [ ]:
results = []
for s in strategies:
    print(f'Running {s["global_combo_id"]}...', flush=True)
    results.append(run_strategy(s, bars=bars))
print('Done.')

## Performance summary

In [ ]:
rows = []
for r in results:
    m = r['metrics']
    rows.append({
        'combo_id': r['combo_id'],
        'n_trades': m['n_trades'],
        'win_rate': m['win_rate'],
        'total_pnl_dollars': m['total_pnl_dollars'],
        'total_return_pct': m['total_return_pct'],
        'sharpe_ratio': m['sharpe_ratio'],
        'max_drawdown_pct': m['max_drawdown_pct'],
        'max_drawdown_dollars': m['max_drawdown_dollars'],
    })
perf = pd.DataFrame(rows)
perf

## Equity curves

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
for r in results:
    eq = r['equity_curve']
    if len(eq) == 0:
        continue
    ax.plot(eq['time'], eq['equity'], label=r['combo_id'], linewidth=1.3)
ax.axhline(STARTING_EQUITY, color='gray', linestyle='--', linewidth=0.8, alpha=0.6)
ax.set_title(f'{BUCKET} — equity curves on 20% test partition')
ax.set_xlabel('time'); ax.set_ylabel('equity ($)')
ax.legend(loc='best', fontsize=9); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## Drawdowns

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
start_time = bars['time'].iloc[0]
for r in results:
    eq = r['equity_curve']
    if len(eq) == 0:
        continue
    eq_full = np.concatenate([[STARTING_EQUITY], eq['equity'].to_numpy()])
    peak = np.maximum.accumulate(eq_full)
    dd_pct = (peak - eq_full) / peak * 100
    times = pd.concat([pd.Series([start_time]), pd.Series(eq['time'].values)]).reset_index(drop=True)
    ax.plot(times, dd_pct, label=r['combo_id'], linewidth=1.3)
ax.set_title(f'{BUCKET} — drawdown (% from peak)')
ax.set_xlabel('time'); ax.set_ylabel('drawdown (%)')
ax.invert_yaxis()
ax.legend(loc='best', fontsize=9); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()